In [1]:
print("hello")

hello


In [2]:
!pip install -q torch torchvision
!pip install -q Pillow numpy
!pip install -q gdown
print("done")



done


In [3]:
import torch
print(f"CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}")

CUDA: True, device: Tesla T4


In [4]:
# import torch

# if torch.cuda.is_available():
#     gpu_name = torch.cuda.get_device_name(0)

#     if "P100" in gpu_name:
#         print("Old GPU detected -> installing compatible PyTorch... -> Restart the system after  ")
#         !pip install torch==2.1.0 torchvision==0.16.0 --quiet

In [5]:
from pathlib import Path

class CONFIG:
    
    # Base
    BASE_DIR = Path("./data")
    DATASET_NAME = "LOL"

    # Download
    FILE_ID = "157bjO1_cFuSd0HWDUuAmcHRJDVyWpOxB"
    ZIP_PATH = BASE_DIR / f"{DATASET_NAME}.zip"
    EXTRACT_PATH = BASE_DIR
    DOWNLOAD_URL = f"https://drive.google.com/uc?id={FILE_ID}"

    # Raw dataset paths
    OUR485_LOW = BASE_DIR / "our485" / "low"
    OUR485_HIGH = BASE_DIR / "our485" / "high"
    EVAL15_LOW = BASE_DIR / "eval15" / "low"

    # Processed dataset paths
    DATASET_DIR = BASE_DIR / "dataset"
    TRAIN_A = DATASET_DIR / "trainA"
    TRAIN_B = DATASET_DIR / "trainB"
    TEST = DATASET_DIR / "test"

In [ ]:
import zipfile
import gdown

# Create base directory
CONFIG.BASE_DIR.mkdir(parents=True, exist_ok=True)

# Download dataset (convert Path → str for gdown)
print("Downloading dataset...")
gdown.download(CONFIG.DOWNLOAD_URL, str(CONFIG.ZIP_PATH), quiet=True)
print("Download complete")

# Extract dataset
with zipfile.ZipFile(CONFIG.ZIP_PATH, 'r') as z:
    z.extractall(CONFIG.EXTRACT_PATH)

# Check structure
print(f"[{CONFIG.BASE_DIR} folder] structure:")
for item in CONFIG.BASE_DIR.iterdir():
    print(item.name)

In [ ]:
low_path = CONFIG.OUR485_LOW
high_path = CONFIG.OUR485_HIGH

low_count = len(list(low_path.iterdir()))
high_count = len(list(high_path.iterdir()))

print("Low images:", low_count)
print("High images:", high_count)

In [ ]:
import shutil

# Create folders
for path in [CONFIG.TRAIN_A, CONFIG.TRAIN_B, CONFIG.TEST]:
    path.mkdir(parents=True, exist_ok=True)

# Copy files
for f in CONFIG.OUR485_LOW.glob("*"):
    shutil.copy(f, CONFIG.TRAIN_A)

for f in CONFIG.OUR485_HIGH.glob("*"):
    shutil.copy(f, CONFIG.TRAIN_B)

for f in CONFIG.EVAL15_LOW.glob("*"):
    shutil.copy(f, CONFIG.TEST)

# Count
print("trainA:", len(list(CONFIG.TRAIN_A.glob("*"))))
print("trainB:", len(list(CONFIG.TRAIN_B.glob("*"))))
print("test  :", len(list(CONFIG.TEST.glob("*"))))

In [ ]:
"""
Two-Stage Low-Light Image Enhancement
Paper: Pre-enhancement (Retinex) + Post-refinement (U-Net with adversarial training)
Supports LOL dataset and custom unpaired datasets.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


# ============================================================
# STAGE 1: PRE-ENHANCEMENT (Retinex-based)
# ============================================================

class AdaptiveToneMapping:
    """
    Implements equation (2)-(4) from the paper.
    Y' = (Lg / Lw) ∘ X
    Lg = log(Lw/Lw_max + 1) / log(Lw_avg/Lw_max + 1)
    """
    @staticmethod
    def get_luminance(image: torch.Tensor) -> torch.Tensor:
        """Convert RGB to luminance map (Lw)."""
        # Standard luminance weights
        weights = torch.tensor([0.2126, 0.7152, 0.0722],
                                device=image.device).view(1, 3, 1, 1)
        return (image * weights).sum(dim=1, keepdim=True).clamp(min=1e-6)

    @staticmethod
    def log_average_luminance(lw: torch.Tensor, sigma: float = 1e-6) -> torch.Tensor:
        """Equation (4): L̄w = exp(1/(m*n) * Σlog(σ + Lw))"""
        return torch.exp(torch.mean(torch.log(sigma + lw), dim=[2, 3], keepdim=True))

    @staticmethod
    def adaptive_tone_map(image: torch.Tensor, lw_max: float = 1.0) -> torch.Tensor:
        """
        Equation (2)-(3): Apply adaptive tone mapping.
        Y' = (Lg / Lw) ∘ X
        """
        lw = AdaptiveToneMapping.get_luminance(image)
        lw_avg = AdaptiveToneMapping.log_average_luminance(lw)

        # Equation (3): Lg = log(Lw/Lw_max + 1) / log(Lw_avg/Lw_max + 1)
        lg = torch.log(lw / lw_max + 1) / (torch.log(lw_avg / lw_max + 1) + 1e-6)

        # Equation (2): Y' = (Lg / Lw) ∘ X
        scale = lg / (lw + 1e-6)
        enhanced = image * scale
        return enhanced.clamp(0, 1)

    @staticmethod
    def pre_enhance(image: torch.Tensor) -> torch.Tensor:
        """Full pre-enhancement pipeline: Retinex decompose → tone map."""
        return AdaptiveToneMapping.adaptive_tone_map(image)


# ============================================================
# STAGE 2: POST-REFINEMENT NETWORK (U-Net based)
# ============================================================

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class RefinementNet(nn.Module):
    """
    U-Net encoder-decoder refinement network.
    Architecture from Table I: conv1-conv5, down1-down4, up1-up4, fusion1-fusion4.
    Input:  Y' — pre-enhanced image (3 ch, 128×128 patches)
    Output: Y  — refined image      (3 ch, 128×128)

    Spatial resolution trace (128-input):
      e1 : 128×128, 32 ch   (conv1)
      e2 : 128×128, 32 ch   (conv2)   ← skip for fusion4
      e3 :  64×64,  32 ch   (down1)   ← skip for fusion3
      e4 :  32×32,  64 ch   (down2)   ← skip for fusion2
      e5 :  16×16, 128 ch   (down3)   ← skip for fusion1
      b  :  16×16, 256 ch   (down4, no extra pool — bottleneck stays at 16)
    Decoder upsample chain: 16→32→64→128→128
    """
    def __init__(self):
        super().__init__()

        # ── Encoder ───────────────────────────────────────────────────────
        self.conv1 = ConvBNReLU(3,   32)   # 128×128
        self.conv2 = ConvBNReLU(32,  32)   # 128×128  → skip s4 (32 ch)
        self.pool  = nn.MaxPool2d(2, 2)

        self.down1 = ConvBNReLU(32,  32)   #  64×64   → skip s3 (32 ch)
        self.down2 = ConvBNReLU(32,  64)   #  32×32   → skip s2 (64 ch)
        self.down3 = ConvBNReLU(64, 128)   #  16×16   → skip s1 (128 ch)

        # Bottleneck — NO extra pool; stays at 16×16
        self.down4 = ConvBNReLU(128, 256)  #  16×16, 256 ch

        # ── Decoder ───────────────────────────────────────────────────────
        # up1: 16×16 → 32×32, fuse with s2 (64 ch)
        self.up1     = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.fusion1 = ConvBNReLU(256 + 64, 128)   #  32×32

        # up2: 32×32 → 64×64, fuse with s3 (32 ch)
        self.up2     = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.fusion2 = ConvBNReLU(128 + 32, 64)    #  64×64

        # up3: 64×64 → 128×128, fuse with s4 (32 ch)
        self.up3     = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.fusion3 = ConvBNReLU(64  + 32, 32)    # 128×128

        # 1×1 output conv
        self.conv5 = nn.Conv2d(32, 3, 1)            # 128×128, 32→3

    def forward(self, x):
        # ── Encoder ───────────────────────────────────────────────────────
        e1 = self.conv1(x)                    # 128×128, 32
        s4 = self.conv2(e1)                   # 128×128, 32  (skip)

        s3 = self.down1(self.pool(s4))        #  64×64,  32  (skip)
        s2 = self.down2(self.pool(s3))        #  32×32,  64  (skip)
        s1 = self.down3(self.pool(s2))        #  16×16, 128  (skip)

        # ── Bottleneck (stays at 16×16) ───────────────────────────────────
        b  = self.down4(s1)                   #  16×16, 256

        # ── Decoder ───────────────────────────────────────────────────────
        d = self.fusion1(torch.cat([self.up1(b), s2], dim=1))   # 32×32
        d = self.fusion2(torch.cat([self.up2(d), s3], dim=1))   # 64×64
        d = self.fusion3(torch.cat([self.up3(d), s4], dim=1))   # 128×128

        return torch.sigmoid(self.conv5(d))   # 128×128, 3


# ============================================================
# DISCRIMINATOR (Relativistic)
# ============================================================

class Discriminator(nn.Module):
    """
    Relativistic discriminator (fully convolutional, handles any input size).
    Used in adversarial loss: l_adv = ((D(Y) - D(Ŷ) - 1)^2 + (D(Ŷ) - D(Y))^2)
    """
    def __init__(self):
        super().__init__()
        def block(in_ch, out_ch, stride=2):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 4, stride=stride, padding=1),
                nn.InstanceNorm2d(out_ch),
                nn.LeakyReLU(0.2, inplace=True)
            )

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            block(64, 128),
            block(128, 256),
            block(256, 512, stride=1),
            nn.Conv2d(512, 1, 4, padding=1)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# LOSS FUNCTIONS
# ============================================================

class VGGPerceptualLoss(nn.Module):
    """l_per = ||φ(Y) - φ(Y')||_2  where φ is VGG feature extractor."""
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=False)
        # Use features up to relu3_3
        self.feature_extractor = nn.Sequential(*list(vgg.features.children())[:16])
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, refined, pre_enhanced):
        return F.mse_loss(self.feature_extractor(refined),
                          self.feature_extractor(pre_enhanced))


class TotalVariationLoss(nn.Module):
    """l_tv = ||∇Y||_1"""
    def forward(self, x):
        diff_h = torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :])
        diff_w = torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1])
        return diff_h.mean() + diff_w.mean()


class RelativisticAdversarialLoss(nn.Module):
    """
    Equation (8): l_adv = ((D(Y) - D(Ŷ) - 1)^2 + (D(Ŷ) - D(Y))^2)
    Y = normal light, Ŷ = refined output
    """
    def forward(self, d_real, d_fake):
        loss_real = ((d_real - d_fake - 1) ** 2).mean()
        loss_fake = ((d_fake - d_real) ** 2).mean()
        return loss_real + loss_fake


class TotalLoss(nn.Module):
    """
    Equation (9): L = l_rec + λ*l_per + μ*l_tv + β*l_adv
    Default: λ=0.1, μ=1, β=1 (from paper)
    """
    def __init__(self, lambda_=0.1, mu=1.0, beta=1.0):
        super().__init__()
        self.lambda_ = lambda_
        self.mu = mu
        self.beta = beta
        self.rec = nn.L1Loss()
        self.per = VGGPerceptualLoss()
        self.tv = TotalVariationLoss()
        self.adv = RelativisticAdversarialLoss()

    def forward(self, refined, pre_enhanced, d_real=None, d_fake=None):
        l_rec = self.rec(refined, pre_enhanced)          # Equation (5)
        l_per = self.per(refined, pre_enhanced)          # Equation (6)
        l_tv = self.tv(refined)                          # Equation (7)

        loss = l_rec + self.lambda_ * l_per + self.mu * l_tv
        if d_real is not None and d_fake is not None:
            l_adv = self.adv(d_real, d_fake)             # Equation (8)
            loss = loss + self.beta * l_adv

        return loss

In [ ]:
import random
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

# # Optional (only if CONFIG exists in same file or importable)
# try:
#     from config import CONFIG
# except:
#     CONFIG = None

VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def _list_images(folder):
    """Return sorted list of valid image filenames."""
    folder = Path(folder)
    return sorted(
        f.name for f in folder.iterdir()
        if f.suffix.lower() in VALID_EXT
    )


# ============================================================
# PAIRED LOL DATASET
# ============================================================

class LOLDataset(Dataset):
    def __init__(self, root: str = None, split: str = 'train', crop_size: int = 128):
        super().__init__()
        assert split in ('train', 'test')

        root = Path(root) if root else (CONFIG.BASE_DIR if CONFIG else Path("./data"))

        subset = 'our485' if split == 'train' else 'eval15'
        self.low_dir   = root / subset / 'low'
        self.high_dir  = root / subset / 'high'

        self.files     = _list_images(self.low_dir)
        self.is_train  = (split == 'train')
        self.crop_size = crop_size
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        low  = Image.open(self.low_dir  / fname).convert('RGB')
        high = Image.open(self.high_dir / fname).convert('RGB')

        if self.is_train:
            i, j, h, w = transforms.RandomCrop.get_params(
                low, output_size=(self.crop_size, self.crop_size))

            low  = transforms.functional.crop(low,  i, j, h, w)
            high = transforms.functional.crop(high, i, j, h, w)

            if random.random() < 0.5:
                low  = transforms.functional.hflip(low)
                high = transforms.functional.hflip(high)

        return self.to_tensor(low), self.to_tensor(high)


# ============================================================
# UNPAIRED DATASET (CycleGAN style)
# ============================================================

class UnpairedDataset(Dataset):
    def __init__(self, root: str = None, crop_size: int = 128, is_train: bool = True):
        super().__init__()

        root = Path(root) if root else (CONFIG.DATASET_DIR if CONFIG else Path("./data/dataset"))

        self.low_dir      = root / 'trainA'
        self.normal_dir   = root / 'trainB'

        self.low_files    = _list_images(self.low_dir)
        self.normal_files = _list_images(self.normal_dir)

        self.is_train  = is_train
        self.crop_size = crop_size
        self.to_tensor = transforms.ToTensor()

        print(f"[Dataset] trainA: {len(self.low_files)} | trainB: {len(self.normal_files)}")

    def __len__(self):
        return len(self.low_files)

    def __getitem__(self, idx):
        low = Image.open(self.low_dir / self.low_files[idx]).convert('RGB')

        nidx   = random.randint(0, len(self.normal_files) - 1)
        normal = Image.open(self.normal_dir / self.normal_files[nidx]).convert('RGB')

        if self.is_train:
            crop_fn = transforms.RandomCrop(self.crop_size)

            low    = crop_fn(low)
            normal = crop_fn(normal)

            if random.random() < 0.5:
                low = transforms.functional.hflip(low)

            if random.random() < 0.5:
                normal = transforms.functional.hflip(normal)

        return self.to_tensor(low), self.to_tensor(normal)


# ============================================================
# TEST DATASET
# ============================================================

class TestDataset(Dataset):
    def __init__(self, test_dir: str = None):
        test_dir = Path(test_dir) if test_dir else (CONFIG.TEST if CONFIG else Path("./data/dataset/test"))

        self.files = sorted(
            p for p in test_dir.iterdir()
            if p.suffix.lower() in VALID_EXT
        )

        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        return self.to_tensor(img), self.files[idx].name

In [ ]:
from pathlib import Path

folders = [CONFIG.TRAIN_A, CONFIG.TRAIN_B, CONFIG.TEST]

for folder in folders:
    ds_store = folder / ".DS_Store"
    
    if ds_store.exists():
        ds_store.unlink()
        print(f"Removed: {ds_store}")

print("Clean!")

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import sys
sys.path.insert(0, '.')
import torch

device = torch.device('cuda')
G = RefinementNet().to(device)
D = Discriminator().to(device)

dummy = torch.randn(2, 3, 128, 128).to(device)
pre   = AdaptiveToneMapping.pre_enhance(dummy)
out   = G(pre)

print(f"G output: {out.shape}")  # expect (2, 3, 128, 128)
print(f"G params: {sum(p.numel() for p in G.parameters()):,}")
print("All good!")

In [ ]:
import time
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm import tqdm

# ================= PATHS (CONFIG) =================
CKPT_DIR = CONFIG.BASE_DIR / "checkpoints"
VIS_DIR  = CONFIG.BASE_DIR / "visuals"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

device  = torch.device('cuda')
ds      = UnpairedDataset(CONFIG.DATASET_DIR, crop_size=128)
loader  = DataLoader(ds, batch_size=64, shuffle=True,
                     num_workers=2, pin_memory=True, drop_last=True)

G       = RefinementNet().to(device)
D       = Discriminator().to(device)
opt_G   = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.9,0.999), weight_decay=1e-4)
opt_D   = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.9,0.999), weight_decay=1e-4)
crit    = TotalLoss(lambda_=0.1, mu=1.0, beta=1.0).to(device)

EPOCHS  = 1000

for epoch in tqdm(range(1, EPOCHS+1), desc="Epochs"):
    G.train(); D.train()
    g_total = d_total = 0.0
    t0 = time.time()

    batch_bar = tqdm(loader, desc=f"Ep {epoch}", leave=False)

    for low, normal in batch_bar:
        low, normal = low.to(device), normal.to(device)

        with torch.no_grad():
            pre = AdaptiveToneMapping.pre_enhance(low)

        # --- Discriminator step ---
        refined = G(pre).detach()
        d_loss  = (((D(normal) - D(refined) - 1) ** 2).mean()
                 + ((D(refined) - D(normal)) ** 2).mean())
        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        # --- Generator step ---
        refined  = G(pre)
        d_r      = D(normal).detach()
        g_loss   = crit(refined, pre, d_r, D(refined))
        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        g_total += g_loss.item()
        d_total += d_loss.item()

        batch_bar.set_postfix(G=f"{g_loss.item():.4f}", D=f"{d_loss.item():.4f}")

    n = len(loader)
    tqdm.write(f"[{epoch:4d}/{EPOCHS}] G={g_total/n:.4f} D={d_total/n:.4f} ({time.time()-t0:.1f}s)")

    if epoch % 50 == 0:
        torch.save(
            {'epoch': epoch, 'G': G.state_dict(), 'D': D.state_dict()},
            CKPT_DIR / f"ckpt_{epoch:04d}.pth"
        )
        G.eval()
        with torch.no_grad():
            s = low[:4]
            p = AdaptiveToneMapping.pre_enhance(s)
            r = G(p)
            save_image(
                torch.cat([s, p, r], dim=0),
                VIS_DIR / f"ep{epoch:04d}.png",
                nrow=4
            )

In [ ]:
import numpy as np

G.eval()
test_ds = LOLDataset(CONFIG.BASE_DIR, split='test')

psnr_list = []

for low, high in test_ds:
    low  = low.unsqueeze(0).to(device)
    high = high.unsqueeze(0).to(device)

    with torch.no_grad():
        pre     = AdaptiveToneMapping.pre_enhance(low)
        refined = G(pre)

    mse = torch.mean((refined - high)**2).item()
    psnr_list.append(10*np.log10(1.0/mse) if mse > 0 else 100)

print(f"PSNR: {np.mean(psnr_list):.3f} dB")

In [ ]:
import matplotlib.pyplot as plt

G.eval()
low, high = test_ds[0]
low_t = low.unsqueeze(0).to(device)

with torch.no_grad():
    pre_t     = AdaptiveToneMapping.pre_enhance(low_t)
    refined_t = G(pre_t)

to_np = lambda t: t.squeeze().permute(1,2,0).cpu().numpy().clip(0,1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, img, title in zip(axes,
    [to_np(low_t), to_np(pre_t), to_np(refined_t), to_np(high.unsqueeze(0).to(device))],
    ['Input (dark)', 'Pre-enhanced', 'Refined (ours)', 'Ground truth']):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()